In [ ]:
import time
from dataclasses import dataclass
from typing import Dict, Tuple, List
import dimod
from neal import SimulatedAnnealingSampler

DEC_WEIGHTS = (1, 2, 3, 3)  # SBE digit weights

def sbe_affine_bits(var: str, J: int, L=0.0, U=1.0) -> Tuple[float, Dict[str, float]]:
    """
    Returns an affine representation:
        x = const + sum_i alpha_i * z_i
    using SBE decimal digits + tail bit.
    """
    const = L
    scale = (U - L)

    coeffs: Dict[str, float] = {}
    for j in range(1, J + 1):
        place = 10 ** (-j)
        for k, w in enumerate(DEC_WEIGHTS, start=1):
            b = f"z_{var}_{j}_{k}"
            coeffs[b] = coeffs.get(b, 0.0) + scale * place * w

    tail = f"z_{var}_tail_J{J}"
    coeffs[tail] = coeffs.get(tail, 0.0) + scale * (10 ** (-J))
    return const, coeffs

def add_square_of_affine(Qlin, Qquad, offset_ref, c0: float, coeffs: Dict[str, float], weight: float = 1.0):
    """
    Add: weight * (c0 + sum_i a_i z_i)^2 to QUBO.
    Uses z_i^2 = z_i.
    """
    # constant
    offset_ref[0] += weight * (c0 * c0)

    items = list(coeffs.items())

    # linear terms: weight*(a_i^2 + 2*c0*a_i)*z_i
    for bi, ai in items:
        Qlin[bi] = Qlin.get(bi, 0.0) + weight * (ai * ai + 2.0 * c0 * ai)

    # quadratic terms: weight*(2*a_i*a_j)*z_i*z_j
    for i in range(len(items)):
        bi, ai = items[i]
        for j in range(i + 1, len(items)):
            bj, aj = items[j]
            u, v = (bi, bj) if bi < bj else (bj, bi)
            Qquad[(u, v)] = Qquad.get((u, v), 0.0) + weight * (2.0 * ai * aj)

def build_bqm(Qlin, Qquad, offset: float) -> dimod.BinaryQuadraticModel:
    return dimod.BinaryQuadraticModel(Qlin, Qquad, offset, vartype=dimod.BINARY)

def qubo_stats(bqm) -> Dict[str, int]:
    return {
        "n_vars": len(bqm.variables),
        "n_quadratic": len(bqm.quadratic),
        "n_linear": len(bqm.linear),
    }

def decode_from_affine(sample: Dict[str, int], const: float, coeffs: Dict[str, float]) -> float:
    val = const
    for b, a in coeffs.items():
        val += a * float(sample.get(b, 0))
    return val

def solve_example1_rolling(
    J0=2, Jstep=2, Jmax=8,
    num_reads=200, sweeps=2500, seed=11,
):
    target1 = 0.1234567
    target2 = 0.7654321

    sampler = SimulatedAnnealingSampler()

    def solve_at(J1, J2):
        Qlin, Qquad = {}, {}
        offset = [0.0]

        c1, a1 = sbe_affine_bits("x1", J1, 0.0, 1.0)
        c2, a2 = sbe_affine_bits("x2", J2, 0.0, 1.0)

        add_square_of_affine(Qlin, Qquad, offset, c1 - target1, a1, weight=1.0)
        add_square_of_affine(Qlin, Qquad, offset, c2 - target2, a2, weight=1.0)

        bqm = build_bqm(Qlin, Qquad, offset[0])

        t0 = time.perf_counter()
        ss = sampler.sample(bqm, num_reads=num_reads, sweeps=sweeps, seed=seed)
        t1 = time.perf_counter()

        best = ss.first.sample
        x1 = decode_from_affine(best, c1, a1)
        x2 = decode_from_affine(best, c2, a2)

        obj = (x1 - target1) ** 2 + (x2 - target2) ** 2

        return {
            "J": (J1, J2),
            "bqm": bqm,
            "time_s": (t1 - t0),
            "x": (x1, x2),
            "obj": obj,
            "stats": qubo_stats(bqm),
        }

    # Rolling loop
    history = []
    J1 = J2 = J0
    rec = solve_at(J1, J2)
    history.append(rec)

    while J1 < Jmax and J2 < Jmax:
        J1 = min(Jmax, J1 + Jstep)
        J2 = min(Jmax, J2 + Jstep)
        rec = solve_at(J1, J2)
        history.append(rec)

    # Monolithic solve at (Jmax, Jmax)
    mono = solve_at(Jmax, Jmax)

    return history, mono

def print_report(history, mono):
    print("\n--- Rolling precision history ---")
    for k, r in enumerate(history):
        print(
            f"it={k:02d}  J={r['J']}  obj={r['obj']:.3e}  "
            f"x=({r['x'][0]:.7f},{r['x'][1]:.7f})  "
            f"nvars={r['stats']['n_vars']}  nquad={r['stats']['n_quadratic']}  time={r['time_s']:.3f}s"
        )
    print("\n--- Monolithic ---")
    print(
        f"J={mono['J']}  obj={mono['obj']:.3e}  "
        f"x=({mono['x'][0]:.7f},{mono['x'][1]:.7f})  "
        f"nvars={mono['stats']['n_vars']}  nquad={mono['stats']['n_quadratic']}  time={mono['time_s']:.3f}s"
    )

if __name__ == "__main__":
    hist, mono = solve_example1_rolling()
    print_report(hist, mono)



--- Rolling precision history ---
it=00  J=(2, 2)  obj=3.281e-05  x=(0.1200000,0.7700000)  nvars=18  nquad=72  time=0.052s
it=01  J=(4, 4)  obj=2.905e-09  x=(0.1235000,0.7654000)  nvars=34  nquad=272  time=0.054s
it=02  J=(6, 6)  obj=1.000e-13  x=(0.1234570,0.7654320)  nvars=50  nquad=600  time=0.086s
it=03  J=(8, 8)  obj=4.950e-32  x=(0.1234567,0.7654321)  nvars=66  nquad=1056  time=0.126s

--- Monolithic ---
J=(8, 8)  obj=4.950e-32  x=(0.1234567,0.7654321)  nvars=66  nquad=1056  time=0.129s
